In [4]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Optional

import mne
import numpy as np
import pandas as pd

# ============================================================
# MULTI-SUBJECT HUP HFO PREPROCESSING
# - runs subject-specific preprocessing for interictal HFO detection
# - reads raw EDF from HUP BIDS
# - drops bad channels from channels.tsv
# - applies notch + CAR
# - creates ripple and fast-ripple epochs
# - NO events.tsv is required for interictal runs
# ============================================================

In [8]:
BIDS_ROOT = Path(r"D:\HUP dataset")

SUBJECT_CONFIGS = [
    {
        "subject": "sub-HUP126",
        "task": "interictal",
        "runs": ["01", "02"],
        "session": "presurgery",
        "acq": "ecog",
        "out_root": Path(r"D:\HUP126_hfo_preproc"),
    },
    {
        "subject": "sub-HUP164",
        "task": "interictal",
        "runs": ["01", "02"],   # adjust if needed
        "session": "presurgery",
        "acq": "seeg",
        "out_root": Path(r"D:\HUP164_hfo_preproc"),
    },
    {
        "subject": "sub-HUP130",
        "task": "interictal",
        "runs": ["01", "02"],   # adjust if needed
        "session": "presurgery",
        "acq": "seeg",
        "out_root": Path(r"D:\HUP130_hfo_preproc"),
    },
    {
        "subject": "sub-HUP157",
        "task": "interictal",
        "runs": ["01", "02"],   # adjust if needed
        "session": "presurgery",
        "acq": "seeg",
        "out_root": Path(r"D:\HUP157_hfo_preproc"),
    },
]

EPOCH_LEN_SEC = 5.0
ARTIFACT_RMS_ZTHRESH = 5.0

# HFO bands
RIPPLE_BAND = (80.0, 240.0)
FAST_BAND = (250.0, 490.0)

In [9]:
def bids_run_prefix(subject: str, session: str, task: str, acq: str, run: str) -> str:
    return f"{subject}_ses-{session}_task-{task}_acq-{acq}_run-{run}"


def read_json(path: Optional[Path]) -> dict:
    if path is None or not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_tsv(path: Optional[Path]) -> pd.DataFrame:
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t")


def extract_bad_channels(ch_df: pd.DataFrame) -> list[str]:
    if ch_df.empty:
        return []
    cols = {c.lower(): c for c in ch_df.columns}
    if "name" not in cols or "status" not in cols:
        return []
    bad_mask = ch_df[cols["status"]].astype(str).str.lower().isin(["bad", "excluded", "reject"])
    return ch_df.loc[bad_mask, cols["name"]].astype(str).tolist()


def robust_z(x: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return 0.6745 * (x - med) / (mad + eps)


def normalize_channel_name(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r"(?i)-?ref$", "", name)
    name = re.sub(r"\s+", "", name)
    return name


def make_fixed_epochs(raw: mne.io.BaseRaw, epoch_len_sec: float) -> mne.Epochs:
    sfreq = float(raw.info["sfreq"])
    n_samp = int(round(epoch_len_sec * sfreq))
    starts = np.arange(0, raw.n_times - n_samp + 1, n_samp, dtype=int)
    if starts.size == 0:
        raise RuntimeError("Recording is shorter than one epoch.")
    events = np.c_[starts, np.zeros_like(starts), np.ones_like(starts)].astype(int)
    return mne.Epochs(
        raw,
        events=events,
        event_id={"segment": 1},
        tmin=0.0,
        tmax=(n_samp - 1) / sfreq,
        baseline=None,
        preload=True,
        reject_by_annotation=False,
        verbose="ERROR",
    )


def artifact_drop_epochs(epo: mne.Epochs, z_thresh: float) -> mne.Epochs:
    data = epo.get_data()  # (n_epochs, n_channels, n_times)
    rms = np.sqrt(np.mean(data ** 2, axis=2))
    keep = np.ones(len(epo), dtype=bool)
    for ci in range(rms.shape[1]):
        z = robust_z(rms[:, ci])
        keep &= np.abs(z) <= z_thresh
    return epo[keep]


def preprocess_one_run(cfg: dict, run: str) -> None:
    subject = cfg["subject"]
    session = cfg["session"]
    task = cfg["task"]
    acq = cfg["acq"]
    out_root = cfg["out_root"]

    prefix = bids_run_prefix(subject, session, task, acq, run)
    ieeg_dir = BIDS_ROOT / subject / f"ses-{session}" / "ieeg"

    edf_path = ieeg_dir / f"{prefix}_ieeg.edf"
    json_path = ieeg_dir / f"{prefix}_ieeg.json"
    channels_path = ieeg_dir / f"{prefix}_channels.tsv"
    electrodes_path = ieeg_dir / f"{subject}_ses-{session}_acq-{acq}_space-fsaveage_electrodes.tsv"

    if not edf_path.exists():
        print(f"[WARN] Missing EDF: {edf_path}")
        return

    out_dir = out_root / subject
    out_dir.mkdir(parents=True, exist_ok=True)

    ripple_out = out_dir / f"{prefix}_ripple_epo.fif"
    fast_out = out_dir / f"{prefix}_fast_epo.fif"
    meta_out = out_dir / f"{prefix}_preproc_meta.json"

    print(f"\n=== PREPROCESSING {prefix} ===")

    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")
    sfreq = float(raw.info["sfreq"])
    print(f"raw sfreq: {sfreq}")

    if sfreq <= 1000.0:
        print("warning: 1024 Hz is only minimally sufficient for FR work; using guard-banded fast band")

    # keep only intracranial channels
    keep_names = [ch for ch, typ in zip(raw.ch_names, raw.get_channel_types()) if typ in ["eeg", "ecog", "seeg"]]
    if keep_names:
        raw.pick(keep_names)

    raw.rename_channels({ch: normalize_channel_name(ch) for ch in raw.ch_names})

    ch_df = read_tsv(channels_path)
    if not ch_df.empty and "name" in ch_df.columns:
        ch_df["name_norm"] = ch_df["name"].map(normalize_channel_name)

    bads = [normalize_channel_name(ch) for ch in extract_bad_channels(ch_df)]
    bads = [ch for ch in bads if ch in raw.ch_names]
    if bads:
        raw.drop_channels(bads)

    print(f"channels after bad-drop: {len(raw.ch_names)}")

    meta = read_json(json_path)
    powerline = float(meta.get("PowerLineFrequency", 60.0))

    # notch harmonics below Nyquist with a small guard band
    notch_freqs = np.arange(powerline, min(480.0, sfreq / 2.0 - 10.0) + 1e-9, powerline)
    if len(notch_freqs):
        raw.notch_filter(freqs=notch_freqs, verbose="ERROR")

    # common average reference
    raw.set_eeg_reference(ref_channels="average", verbose="ERROR")

    ripple = raw.copy().filter(
        l_freq=RIPPLE_BAND[0],
        h_freq=RIPPLE_BAND[1],
        fir_design="firwin",
        verbose="ERROR",
    )
    fast = raw.copy().filter(
        l_freq=FAST_BAND[0],
        h_freq=FAST_BAND[1],
        fir_design="firwin",
        verbose="ERROR",
    )

    ripple_epo = make_fixed_epochs(ripple, EPOCH_LEN_SEC)
    fast_epo = make_fixed_epochs(fast, EPOCH_LEN_SEC)

    ripple_epo = artifact_drop_epochs(ripple_epo, ARTIFACT_RMS_ZTHRESH)
    fast_epo = artifact_drop_epochs(fast_epo, ARTIFACT_RMS_ZTHRESH)

    # keep only epochs surviving in both bands
    ripple_starts = ripple_epo.events[:, 0]
    fast_starts = fast_epo.events[:, 0]
    common = np.intersect1d(ripple_starts, fast_starts)

    ripple_epo = ripple_epo[np.isin(ripple_starts, common)]
    fast_epo = fast_epo[np.isin(fast_starts, common)]

    ripple_epo.save(ripple_out, overwrite=True)
    fast_epo.save(fast_out, overwrite=True)

    elec_df = read_tsv(electrodes_path)
    if not elec_df.empty and "name" in elec_df.columns:
        elec_df["name_norm"] = elec_df["name"].map(normalize_channel_name)
        elec_df = elec_df[elec_df["name_norm"].isin(raw.ch_names)].copy()
        elec_keep_path = out_dir / f"{subject}_electrodes_kept.tsv"
        elec_df.to_csv(elec_keep_path, sep="\t", index=False)
    else:
        elec_keep_path = None

    meta_to_save = {
        "subject": subject,
        "session": session,
        "task": task,
        "acquisition": acq,
        "run": run,
        "raw_sfreq": sfreq,
        "powerline_frequency": powerline,
        "n_channels_after_drop": len(raw.ch_names),
        "ripple_band": RIPPLE_BAND,
        "fast_band": FAST_BAND,
        "epoch_len_sec": EPOCH_LEN_SEC,
        "artifact_rms_z_thresh": ARTIFACT_RMS_ZTHRESH,
        "n_ripple_epochs": len(ripple_epo),
        "n_fast_epochs": len(fast_epo),
        "electrodes_tsv_kept": str(elec_keep_path) if elec_keep_path else None,
        "note": "interictal run; no events.tsv required for HFO preprocessing",
    }

    with open(meta_out, "w", encoding="utf-8") as f:
        json.dump(meta_to_save, f, indent=2)

    print(f"saved: {ripple_out}")
    print(f"saved: {fast_out}")
    print(f"kept common epochs: {len(ripple_epo)}")


In [10]:
def main():
    for cfg in SUBJECT_CONFIGS:
        print("\n" + "=" * 70)
        print(f"SUBJECT: {cfg['subject']}")
        print("=" * 70)

        for run in cfg["runs"]:
            preprocess_one_run(cfg, run)

    print("\nDone preprocessing.")


if __name__ == "__main__":
    main()


SUBJECT: sub-HUP126

=== PREPROCESSING sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-01 ===
raw sfreq: 1024.0
channels after bad-drop: 125
saved: D:\HUP126_hfo_preproc\sub-HUP126\sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-01_ripple_epo.fif
saved: D:\HUP126_hfo_preproc\sub-HUP126\sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-01_fast_epo.fif
kept common epochs: 26

=== PREPROCESSING sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-02 ===
raw sfreq: 1024.0
channels after bad-drop: 125
saved: D:\HUP126_hfo_preproc\sub-HUP126\sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-02_ripple_epo.fif
saved: D:\HUP126_hfo_preproc\sub-HUP126\sub-HUP126_ses-presurgery_task-interictal_acq-ecog_run-02_fast_epo.fif
kept common epochs: 32

SUBJECT: sub-HUP164

=== PREPROCESSING sub-HUP164_ses-presurgery_task-interictal_acq-seeg_run-01 ===
raw sfreq: 1024.0
channels after bad-drop: 176
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.